# ClimateScope Bangladesh & South Asia
## Phase 1 — Data Collection & Cleaning

**Author:** Shamsul AL Mazid | GitHub: [almazid82](https://github.com/almazid82)

**Objective:** Collect, validate, and clean all climate, economic, and disaster datasets
needed for the full multi-phase analysis. Every dataset is saved to `data/raw/` and
cleaned versions to `data/processed/`.

### Data Sources Used in This Notebook

| # | Dataset | Source | Coverage |
|---|---------|--------|----------|
| 1 | Bangladesh Temperature & Rainfall | NASA POWER API | 1984–2023 |
| 2 | Global Temperature Anomaly | NASA GISS (local) | 1880–2024 |
| 3 | Global CO₂ Emissions | NOAA (local) | 2000–2016 |
| 4 | Bangladesh Economic Indicators | World Bank API | 1960–2023 |
| 5 | Global Sea Level Rise | CSIRO / GitHub Datasets | 1880–2013 |
| 6 | Bangladesh Disaster Data | EM-DAT *(awaiting access)* | 1960–2023 |

---

## 1. Import Libraries & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import requests
import warnings
import os
import wbgapi as wb

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.3f}'.format)

# ── Paths ─────────────────────────────────────────────────────────────────────
RAW  = "../data/raw"
PROC = "../data/processed"
FIG  = "../outputs/figures"

for path in [RAW, PROC, FIG]:
    os.makedirs(path, exist_ok=True)

print("✅ Libraries loaded.")
print(f"   Raw data  → {os.path.abspath(RAW)}")
print(f"   Processed → {os.path.abspath(PROC)}")

## 2. Bangladesh Climate Data — NASA POWER API

**Source:** [NASA POWER](https://power.larc.nasa.gov/) — Prediction of Worldwide Energy Resources
**Why:** Provides station-quality gridded meteorological data for any coordinates.
Bangladesh centre: **23.685°N, 90.356°E**

**Parameters downloaded:**
- `T2M` — Temperature at 2 metres (°C)
- `PRECTOTCORR` — Monthly precipitation (mm/day)
- `RH2M` — Relative humidity at 2 metres (%)

In [ ]:
def fetch_nasa_power(lat, lon, start_year, end_year, parameters):
    """Download monthly climate data from NASA POWER API for a given location."""
    url = "https://power.larc.nasa.gov/api/temporal/monthly/point"
    params = {
        "parameters": ",".join(parameters),
        "community":  "RE",
        "longitude":  lon,
        "latitude":   lat,
        "start":      start_year,
        "end":        end_year,
        "format":     "JSON"
    }
    print(f"Downloading from NASA POWER API... (lat={lat}, lon={lon}, {start_year}–{end_year})")
    resp = requests.get(url, params=params, timeout=90)
    resp.raise_for_status()
    data = resp.json()["properties"]["parameter"]

    records = {}
    for param, monthly_data in data.items():
        for yyyymm, value in monthly_data.items():
            records.setdefault(yyyymm, {})[param] = value

    df = pd.DataFrame.from_dict(records, orient="index")
    df.index = pd.to_datetime(df.index, format="%Y%m")
    df.index.name = "Date"
    df = df.sort_index()
    df.replace(-999.0, np.nan, inplace=True)   # NASA missing-value code
    return df

# Download Bangladesh climate data ─────────────────────────────────────────────
BGD_LAT, BGD_LON = 23.685, 90.356
PARAMS = ["T2M", "PRECTOTCORR", "RH2M"]

df_nasa = fetch_nasa_power(BGD_LAT, BGD_LON, 1984, 2023, PARAMS)
df_nasa.columns = ["Temperature_C", "Precipitation_mm_day", "Humidity_pct"]

df_nasa.to_csv(f"{RAW}/bangladesh_nasa_power_monthly.csv")
print(f"\n✅ Saved: bangladesh_nasa_power_monthly.csv")
print(f"   Shape: {df_nasa.shape} | Period: {df_nasa.index[0].date()} → {df_nasa.index[-1].date()}")
df_nasa.head(8)

In [ ]:
# Quick validation plot
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(df_nasa.index, df_nasa["Temperature_C"], color="tomato", linewidth=0.8)
axes[0].set_ylabel("Temp (°C)"); axes[0].set_title("Bangladesh Monthly Temperature (NASA POWER)")

axes[1].bar(df_nasa.index, df_nasa["Precipitation_mm_day"], color="steelblue", width=20)
axes[1].set_ylabel("Precip (mm/day)"); axes[1].set_title("Bangladesh Monthly Precipitation")

axes[2].plot(df_nasa.index, df_nasa["Humidity_pct"], color="green", linewidth=0.8)
axes[2].set_ylabel("Humidity (%)"); axes[2].set_title("Bangladesh Monthly Relative Humidity")

plt.tight_layout()
plt.savefig(f"{FIG}/phase1_bangladesh_nasa_overview.png", dpi=150)
plt.show()
print("Raw data looks good — seasonal patterns clearly visible.")

### Key Findings — NASA POWER Data
- Clear **monsoon signal** in precipitation (June–September peaks)
- Temperature ranges ~15°C (winter) to ~33°C (summer)
- Humidity highest during monsoon months (June–September)
- No major data gaps detected

## 3. Global Temperature Anomaly — NASA GISS

**Source:** [NASA GISS Surface Temperature Analysis](https://data.giss.nasa.gov/gistemp/) (GISTEMP v4)
**Why:** Provides the global baseline — we compare Bangladesh local warming against global trend.
**File:** `GLB.Ts+dSST.csv` (already downloaded to `data/raw/`)

In [ ]:
df_giss = pd.read_csv(f"{RAW}/GLB.Ts+dSST.csv", skiprows=1)
df_giss = df_giss[["Year", "J-D", "DJF", "MAM", "JJA", "SON"]]
df_giss.columns = ["Year", "Annual_Anomaly_C", "Winter", "Spring", "Summer", "Autumn"]

df_giss.replace("***", np.nan, inplace=True)
df_giss = df_giss.apply(lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x))
df_giss.iloc[:, 1:] = df_giss.iloc[:, 1:].apply(pd.to_numeric, errors="coerce")
df_giss["Year"] = df_giss["Year"].astype(int)

df_giss.fillna(df_giss.median(numeric_only=True), inplace=True)
df_giss.dropna(inplace=True)

df_giss.to_csv(f"{PROC}/global_temperature_anomaly.csv", index=False)
print(f"✅ NASA GISS cleaned | Shape: {df_giss.shape} | Years: {df_giss.Year.min()}–{df_giss.Year.max()}")
print(f"   Missing values: {df_giss.isnull().sum().sum()}")
df_giss.tail(5)

## 4. Global CO₂ Emissions Dataset

**Source:** NOAA / Our World in Data
**Why:** CO₂ is the primary driver of anthropogenic warming — essential for correlation analysis.

In [ ]:
df_co2 = pd.read_csv(f"{RAW}/CO2_emissions_global.csv")
print("Columns:", df_co2.columns.tolist())
print("Shape:", df_co2.shape)
df_co2.head()

In [ ]:
# Clean and standardise
df_co2.columns = [c.strip() for c in df_co2.columns]

# Detect Year column
year_col = [c for c in df_co2.columns if "year" in c.lower() or "Year" in c][0]
df_co2.rename(columns={year_col: "Year"}, inplace=True)

# Convert Year to int if needed
if df_co2["Year"].dtype == object:
    df_co2["Year"] = pd.to_numeric(df_co2["Year"], errors="coerce")
df_co2.dropna(subset=["Year"], inplace=True)
df_co2["Year"] = df_co2["Year"].astype(int)

# Rename numeric columns
num_cols = [c for c in df_co2.columns if c != "Year"]
if num_cols:
    df_co2.rename(columns={num_cols[0]: "CO2_emissions"}, inplace=True)

df_co2.to_csv(f"{PROC}/co2_emissions_clean.csv", index=False)
print(f"✅ CO₂ data cleaned | Shape: {df_co2.shape} | Years: {df_co2.Year.min()}–{df_co2.Year.max()}")
df_co2.tail(5)

## 5. Bangladesh Economic Indicators — World Bank API

**Source:** [World Bank Open Data](https://data.worldbank.org/) via `wbgapi`
**Country code:** BGD (Bangladesh)

| Indicator Code | Meaning |
|----------------|---------|
| `NY.GDP.MKTP.CD` | GDP (current US$) |
| `NY.GDP.PCAP.CD` | GDP per capita (current US$) |
| `SP.POP.TOTL` | Total population |
| `AG.LND.AGRI.ZS` | Agricultural land (% of total) |
| `EN.ATM.CO2E.KT` | CO₂ emissions (kt) — Bangladesh only |
| `SH.DYN.MORT` | Under-5 mortality rate (proxy for development) |

In [ ]:
INDICATORS = {
    "NY.GDP.MKTP.CD": "GDP_USD",
    "NY.GDP.PCAP.CD": "GDP_per_capita_USD",
    "SP.POP.TOTL":    "Population",
    "AG.LND.AGRI.ZS": "Agricultural_land_pct",
    "EN.ATM.CO2E.KT": "CO2_kt_BGD",
    "SH.DYN.MORT":    "Under5_mortality",
}

print("Downloading from World Bank API...")
try:
    raw_wb = wb.data.DataFrame(
        list(INDICATORS.keys()),
        economy="BGD",
        time=range(1960, 2024),
        skipBlanks=True,
        columns="series"
    )
    df_wb = raw_wb.reset_index()

    # Rename columns
    df_wb.rename(columns={"time": "Year"}, inplace=True)
    df_wb["Year"] = df_wb["Year"].astype(str).str.extract(r"(\d{4})").astype(int)
    df_wb.rename(columns=INDICATORS, inplace=True)

    df_wb = df_wb.sort_values("Year").reset_index(drop=True)
    df_wb.to_csv(f"{PROC}/bangladesh_worldbank_indicators.csv", index=False)
    print(f"✅ World Bank data downloaded | Shape: {df_wb.shape} | Years: {df_wb.Year.min()}–{df_wb.Year.max()}")
    df_wb.tail(8)

except Exception as e:
    print(f"⚠️  World Bank API error: {e}")
    print("Creating placeholder — re-run when connected to internet.")
    df_wb = pd.DataFrame({"Year": range(1960, 2024)})

In [ ]:
# Missing value audit for World Bank data
print("Missing values per indicator:")
missing = df_wb.isnull().sum()
pct = (df_wb.isnull().mean() * 100).round(1)
summary = pd.DataFrame({"Missing Count": missing, "Missing %": pct})
print(summary[summary["Missing Count"] > 0])

# Forward-fill + linear interpolation for economic indicators
df_wb_clean = df_wb.set_index("Year").interpolate(method="linear").reset_index()
df_wb_clean.to_csv(f"{PROC}/bangladesh_worldbank_clean.csv", index=False)
print(f"\n✅ World Bank cleaned and saved.")

### Key Findings — World Bank Data
- Bangladesh GDP has grown **~35x** since 1990 — rapid development context
- Agricultural land declining — urbanisation and industrialisation trend
- Under-5 mortality falling sharply — development gains, yet climate vulnerability remains high

## 6. Global Sea Level Rise — CSIRO Dataset

**Source:** Church & White (2011) via [GitHub Datasets](https://github.com/datasets/sea-level-rise)
**Why:** Bangladesh has 710 km of coastline — sea level rise directly threatens 30M+ coastal people.

In [ ]:
SEA_URL = (
    "https://raw.githubusercontent.com/datasets/sea-level-rise"
    "/master/data/epa-sea-level.csv"
)

print("Downloading sea level dataset...")
try:
    df_sea = pd.read_csv(SEA_URL)
    df_sea.columns = [c.strip() for c in df_sea.columns]
    print("Columns:", df_sea.columns.tolist())

    # Standardise
    year_col = [c for c in df_sea.columns if "year" in c.lower()][0]
    df_sea.rename(columns={year_col: "Year"}, inplace=True)
    df_sea["Year"] = df_sea["Year"].apply(
        lambda x: int(str(x).split(".")[0]) if pd.notnull(x) else np.nan
    )

    level_col = [c for c in df_sea.columns
                 if any(k in c.lower() for k in ["csiro", "adjusted", "sea level"])][0]
    df_sea.rename(columns={level_col: "Sea_Level_mm"}, inplace=True)
    df_sea = df_sea[["Year", "Sea_Level_mm"]].dropna()

    df_sea.to_csv(f"{RAW}/sea_level_rise.csv", index=False)
    df_sea.to_csv(f"{PROC}/sea_level_rise_clean.csv", index=False)
    print(f"✅ Sea level data saved | Shape: {df_sea.shape} | Years: {df_sea.Year.min()}–{df_sea.Year.max()}")
    df_sea.tail(5)

except Exception as e:
    print(f"⚠️  Download failed: {e}")
    df_sea = pd.DataFrame({"Year": [], "Sea_Level_mm": []})

In [ ]:
# Sea level trend plot
if not df_sea.empty:
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(df_sea["Year"], df_sea["Sea_Level_mm"], color="navy", linewidth=1.5)
    ax.fill_between(df_sea["Year"], df_sea["Sea_Level_mm"],
                    df_sea["Sea_Level_mm"].min(), alpha=0.15, color="navy")
    ax.set_xlabel("Year"); ax.set_ylabel("Sea Level Change (mm)")
    ax.set_title("Global Mean Sea Level Rise — Church & White (2011)")
    ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
    plt.tight_layout()
    plt.savefig(f"{FIG}/phase1_sea_level_trend.png", dpi=150)
    plt.show()

## 7. Bangladesh Disaster Data — EM-DAT (Placeholder)

**Source:** [EM-DAT — The International Disaster Database](https://www.emdat.be/)
**Registration:** Free at emdat.be → request access → download Bangladesh disasters CSV

**How to download (after registration):**
1. Login → Country Profiles → Bangladesh
2. Filter: Disaster Type = Flood | Period = 1960–2023
3. Export as CSV → save to `data/raw/emdat_bangladesh_disasters.csv`

**Expected columns:** Year, Disaster Type, Total Deaths, No Affected, Total Damages (000 US$)

In [ ]:
EMDAT_PATH = f"{RAW}/emdat_bangladesh_disasters.csv"

if os.path.exists(EMDAT_PATH):
    df_emdat = pd.read_csv(EMDAT_PATH)
    print(f"✅ EM-DAT loaded | Shape: {df_emdat.shape}")
    print("Columns:", df_emdat.columns.tolist())

    # Standardise key columns
    rename_map = {
        "Year": "Year",
        "Total Deaths": "Deaths",
        "No Affected": "Affected",
        "Total Damages, Adjusted ('000 US$)": "Damage_000USD",
        "Disaster Type": "Disaster_Type",
    }
    df_emdat = df_emdat.rename(columns={k: v for k, v in rename_map.items() if k in df_emdat.columns})

    flood_cols = ["Year", "Disaster_Type", "Deaths", "Affected", "Damage_000USD"]
    available = [c for c in flood_cols if c in df_emdat.columns]
    df_emdat = df_emdat[available]
    df_emdat.to_csv(f"{PROC}/emdat_bangladesh_clean.csv", index=False)
    print(f"✅ EM-DAT cleaned and saved.")
    df_emdat.head()
else:
    print("⏳ EM-DAT file not yet downloaded.")
    print(f"   When ready, place CSV at: {os.path.abspath(EMDAT_PATH)}")
    print("   Then re-run this cell — code will load automatically.")
    df_emdat = None

## 8. Build Master Annual Dataset

Aggregate all monthly data to annual means, then merge everything by **Year** into a single master dataset for Phase 2 analysis.

In [ ]:
# Aggregate NASA POWER monthly → annual
df_nasa_annual = df_nasa.resample("YE").mean()
df_nasa_annual.index = df_nasa_annual.index.year
df_nasa_annual.index.name = "Year"
df_nasa_annual.columns = ["BGD_Temp_C", "BGD_Precip_mm_day", "BGD_Humidity_pct"]
df_nasa_annual = df_nasa_annual.reset_index()

print("NASA POWER annual shape:", df_nasa_annual.shape)
df_nasa_annual.head(3)

In [ ]:
# Merge all datasets on Year
master = df_nasa_annual.copy()
master = master.merge(df_giss[["Year", "Annual_Anomaly_C"]], on="Year", how="left")
master = master.merge(df_wb_clean, on="Year", how="left")
master = master.merge(df_sea, on="Year", how="left")

# Add CO2 if Year column matches
if not df_co2.empty and "CO2_emissions" in df_co2.columns:
    master = master.merge(df_co2[["Year", "CO2_emissions"]], on="Year", how="left")

master = master.sort_values("Year").reset_index(drop=True)
master.to_csv(f"{PROC}/master_annual_dataset.csv", index=False)

print(f"✅ Master dataset saved")
print(f"   Shape: {master.shape}")
print(f"   Years: {master.Year.min()} → {master.Year.max()}")
print(f"   Columns ({len(master.columns)}): {master.columns.tolist()}")

In [ ]:
# Missing value heatmap
fig, ax = plt.subplots(figsize=(14, 5))
missing_pct = master.isnull().mean() * 100
colors = ["#2ecc71" if v < 10 else "#f39c12" if v < 30 else "#e74c3c" for v in missing_pct]
bars = ax.bar(missing_pct.index, missing_pct.values, color=colors)
ax.set_xticklabels(missing_pct.index, rotation=45, ha="right")
ax.set_ylabel("Missing %")
ax.set_title("Data Completeness of Master Dataset (Green < 10% missing)")
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.axhline(10, color="orange", linewidth=1, linestyle="--", label="10% threshold")
ax.axhline(30, color="red", linewidth=1, linestyle="--", label="30% threshold")
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG}/phase1_data_completeness.png", dpi=150)
plt.show()

## 9. Phase 1 Summary & Data Quality Report

In [ ]:
print("=" * 60)
print("   PHASE 1 COMPLETE — DATA COLLECTION & CLEANING")
print("=" * 60)

datasets = {
    "Bangladesh Climate (NASA POWER)": df_nasa.shape,
    "Global Temp Anomaly (NASA GISS)": df_giss.shape,
    "CO₂ Emissions (Global)":          df_co2.shape,
    "World Bank Economic Indicators":   df_wb_clean.shape,
    "Sea Level Rise (CSIRO)":           df_sea.shape,
    "MASTER Annual Dataset":            master.shape,
}

for name, shape in datasets.items():
    status = "✅" if shape[0] > 0 else "⏳"
    print(f"   {status}  {name:<40} {shape[0]} rows × {shape[1]} cols")

emdat_status = "✅ Loaded" if df_emdat is not None else "⏳ Awaiting EM-DAT registration"
print(f"\n   EM-DAT Disaster Data: {emdat_status}")

print("\n" + "─" * 60)
print("   Files saved to data/processed/:")
for f in os.listdir(PROC):
    size_kb = os.path.getsize(f"{PROC}/{f}") // 1024
    print(f"   📄  {f:<45} {size_kb} KB")
print("─" * 60)
print("\n   ➡  Next: Open 02_exploratory_data_analysis.ipynb")

### Key Findings — Phase 1

| Finding | Detail |
|---------|--------|
| Bangladesh temp data | 40 years of monthly records (1984–2023) — strong monsoon signal |
| Global warming baseline | NASA GISS confirms +1.2°C anomaly by 2023 vs 1951–1980 baseline |
| Bangladesh GDP | Grew from ~$4B (1960) to ~$460B (2023) — rapid development |
| Sea level | ~200mm rise since 1880 — accelerating post-1990 |
| Data gaps | World Bank some indicators sparse pre-1972 — handled by interpolation |
| EM-DAT | Registration pending — re-run cell 7 when CSV is downloaded |

---

**→ Phase 2 will use `data/processed/master_annual_dataset.csv` for full EDA**
*Author: Shamsul AL Mazid | github.com/almazid82*